# RAG 知识库问答系统 — Demo Notebook

本 Notebook 演示如何使用 RAG 系统的核心模块进行文档摄入和问答。

> 确保已配置 `.env` 文件中的 API Key

In [ ]:
import sys
sys.path.insert(0, '..')

from src.pipeline.pipeline import RAGPipeline
from src.config import settings

print(f"当前模型: {settings.model_name}")
print(f"Embedding 模型: {settings.embedding_model}")
print(f"向量存储路径: {settings.chroma_persist_dir}")

In [ ]:
# 初始化 RAG 流水线
pipeline = RAGPipeline()
print(f"当前知识库文档切片总数: {pipeline.document_count}")

## 1. 摄入文档

将 `demo_data/sample_knowledge.md` 导入知识库

In [ ]:
import os

sample_file = os.path.abspath('../demo_data/sample_knowledge.md')
print(f"导入文件: {sample_file}")

result = pipeline.ingest(sample_file)

if result['success']:
    doc = result['document']
    print(f"✅ 导入成功!")
    print(f"   文件名: {doc['filename']}")
    print(f"   切片数: {result['chunk_count']}")
else:
    print(f"❌ 导入失败: {result['error']}")

In [ ]:
# 查看已导入的文档
docs = pipeline.list_documents()
for doc in docs:
    print(f"📄 {doc['source']} — {doc['chunk_count']} 个片段")

## 2. RAG 问答

基于已导入的文档进行提问

In [ ]:
# 提问
question = "Python 有哪些主要特点？"
result = pipeline.query(question)

print(f"🤔 问题: {question}")
print(f"📝 回答:\n{result['answer']}")
print(f"\n📚 引用来源 ({len(result['sources'])} 个):")
for i, s in enumerate(result['sources'], 1):
    print(f"  {i}. {s['source']} (相关度: {s['score']:.2%})")

In [ ]:
# 再问一个问题
question2 = "Python 中如何定义类和继承？请给出代码示例"
result2 = pipeline.query(question2)

print(f"🤔 问题: {question2}")
print(f"📝 回答:\n{result2['answer']}")
print(f"\n📚 引用来源 ({len(result2['sources'])} 个):")
for i, s in enumerate(result2['sources'], 1):
    print(f"  {i}. {s['source']} (相关度: {s['score']:.2%})")

In [ ]:
# 测试：问一个文档中没有的问题
question3 = "今天北京的天气怎么样？"
result3 = pipeline.query(question3)

print(f"🤔 问题: {question3}")
print(f"📝 回答:\n{result3['answer']}")
print(f"\n📚 引用来源: {len(result3['sources'])} 个")

## 3. 流式输出演示

In [ ]:
# 流式输出（在 Notebook 中也能看到逐字生成的效果）
question = "解释一下 RAG 是什么"
print(f"🤔 问题: {question}\n📝 回答: ", end="", flush=True)

async def demo_stream():
    async for chunk in pipeline.query_stream(question):
        print(chunk, end="", flush=True)

import asyncio
await demo_stream()

## 4. 文档管理

In [ ]:
# 删除文档（如需重新导入）
# pipeline.delete_document("sample_knowledge.md")
# print("文档已删除")
# print(f"当前切片数: {pipeline.document_count}")

## 5. 模块级使用

如果需要更精细的控制，可以直接使用各个子模块

In [ ]:
# 直接使用检索器
from src.retrieval.retriever import DocumentRetriever

retriever = DocumentRetriever()
results = retriever.retrieve("Python 函数定义", top_k=3)

for i, r in enumerate(results, 1):
    print(f"{i}. [{r['source']}] (相关度: {r['score']:.2%})")
    print(f"   {r['content'][:100]}...")
    print()